# Cognix Round 2: TRIBE Inference (Overnight Run)

**Purpose:** Run TRIBE v2 on all unique texts and cache raw tensors + pooled vectors to Google Drive.

**Setup:**
1. Add HuggingFace token as Colab secret (`HF_TOKEN`)
2. Runtime > Change runtime type > **A100 GPU + High RAM**
3. Run all cells
4. Go to sleep — this takes ~13 hours

**Resumable:** If Colab disconnects, just re-run. Completed texts are skipped automatically.

In [ ]:
# ---- System dependencies ----
!apt-get update -qq && apt-get install -y -qq ffmpeg

# ---- Install dependencies ----
!pip uninstall -y numpy
!pip install 'numpy>=1.26.4,<2.1.0'

!git clone https://github.com/facebookresearch/tribev2.git /content/tribev2-repo
!pip install -e '/content/tribev2-repo/.[plotting]'
!pip install torchcodec==0.2.1

# ---- Clone Cognix repo ----
!git clone https://github.com/gdavid7/cognix.git /content/cognix

# ---- HuggingFace auth ----
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# NOTE: If Colab asks to restart after numpy, click Restart,
# then skip this cell and run from the next cell onward.

In [ ]:
# ---- Mount Google Drive for persistent storage ----
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/cognix_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache directory: {CACHE_DIR}')

In [ ]:
import json
import sys
import time
import hashlib
import tempfile
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ---- Load unique texts ----
TEXTS_PATH = '/content/cognix/data/r2_unique_texts.jsonl'

texts = []
with open(TEXTS_PATH) as f:
    for line in f:
        entry = json.loads(line)
        texts.append(entry)

print(f'Total unique texts: {len(texts)}')

In [ ]:
# ---- Check what's already cached ----
raw_dir = Path(CACHE_DIR) / 'raw_tensors'
pooled_dir = Path(CACHE_DIR) / 'pooled_vectors'
raw_dir.mkdir(exist_ok=True)
pooled_dir.mkdir(exist_ok=True)

already_done = set()
for entry in texts:
    h = entry['hash']
    if (pooled_dir / f'{h}.npy').exists():
        already_done.add(h)

remaining = [e for e in texts if e['hash'] not in already_done]
print(f'Already cached: {len(already_done)}')
print(f'Remaining: {len(remaining)}')
print(f'Estimated time: ~{len(remaining) * 38 / 3600:.1f} hours on A100')

In [ ]:
# ---- Load TRIBE v2 ----
from tribev2 import TribeModel

tribe_model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='./cache')
print('TRIBE v2 loaded.')

In [ ]:
def text_to_raw_tensor(text):
    """Run TRIBE on text, return raw brain tensor (T, 20484)."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
        tmp_path = f.name
    try:
        events = tribe_model.get_events_dataframe(text_path=tmp_path)
        preds, _segments = tribe_model.predict(events=events)
        return np.asarray(preds)
    finally:
        os.unlink(tmp_path)

In [ ]:
# ---- Main inference loop (resumable) ----
# Saves after every text. If Colab disconnects, re-run and it picks up where it left off.

index_path = Path(CACHE_DIR) / 'text_index.json'
if index_path.exists():
    with open(index_path) as f:
        text_index = json.load(f)
else:
    text_index = {}

failed = []
start_time = time.time()

for i, entry in enumerate(tqdm(remaining, desc='TRIBE inference')):
    h = entry['hash']
    text = entry['text']

    # Double-check not already done (in case of re-run)
    if (pooled_dir / f'{h}.npy').exists():
        continue

    try:
        # Run TRIBE
        raw = text_to_raw_tensor(text)
        pooled = raw.mean(axis=0)

        # Save raw tensor
        np.savez_compressed(raw_dir / f'{h}.npz', tensor=raw)

        # Save pooled vector
        np.save(pooled_dir / f'{h}.npy', pooled)

        # Update index
        text_index[h] = text
        if i % 10 == 0:  # save index every 10 texts
            with open(index_path, 'w') as f:
                json.dump(text_index, f)

    except Exception as e:
        failed.append({'hash': h, 'text': text[:100], 'error': str(e)})
        print(f'FAILED [{h}]: {text[:60]}... -> {e}')

# Final index save
with open(index_path, 'w') as f:
    json.dump(text_index, f)

elapsed = time.time() - start_time
print(f'\nDone. Processed {len(remaining) - len(failed)} texts in {elapsed/3600:.1f} hours.')
print(f'Failed: {len(failed)}')
if failed:
    for f_entry in failed:
        print(f"  {f_entry['hash']}: {f_entry['text']}... -> {f_entry['error']}")

In [ ]:
# ---- Verify ----
n_raw = len(list(raw_dir.glob('*.npz')))
n_pooled = len(list(pooled_dir.glob('*.npy')))
print(f'Raw tensors on Drive: {n_raw}')
print(f'Pooled vectors on Drive: {n_pooled}')
print(f'Text index entries: {len(text_index)}')
print(f'\nTotal unique texts requested: {len(texts)}')
print(f'Coverage: {n_pooled}/{len(texts)} ({100*n_pooled/len(texts):.1f}%)')

# Check a sample
sample_hash = texts[0]['hash']
sample_pooled = np.load(pooled_dir / f'{sample_hash}.npy')
sample_raw = np.load(raw_dir / f'{sample_hash}.npz')['tensor']
print(f'\nSample pooled shape: {sample_pooled.shape}')
print(f'Sample raw shape: {sample_raw.shape}')
print(f'\nAll cached to: {CACHE_DIR}')
print('Download pooled_vectors/ to your Mac for local analysis.')